In [1]:
import numpy as np
from pyscf import gto, scf, cc

xyzfile = "/home/yichi/research/w4_17/w4-17_all/w4_17_xyz/closed_shell/h2o.xyz"
with open(xyzfile, "r") as f:
    lines = f.readlines()
    second_line = lines[1]
    charge = int(second_line.split("charge=")[1].split()[0])
    mult   = int(second_line.split("mult=")[1].split()[0])
    spin   = mult - 1
    atoms = "".join(lines[2:])

mol = gto.M(atom = atoms,
            basis = 'ccpvdz',
            verbose=4,
            unit='angstrom',
            symmetry=0,
            charge=charge,
            spin=spin,
            max_memory=40000)

mf = scf.RHF(mol)
mf.max_cycle = 100
mf.kernel()

stable = False
while not stable:
    mo_i, _, stable, _ = mf.stability(return_status=True)
    dm = mf.make_rdm1(mo_i, mf.mo_occ)
    mf.kernel(dm0=dm)

mycc = cc.CCSD(mf)
mycc.set_frozen()
mycc.kernel()
et = mycc.ccsd_t()

print(f"E SCF-HF = {mf.e_tot}")
print(f"E CCSD   = {mycc.e_tot}")
print(f"E CCSD(T) = {mycc.e_tot+et}")

System: uname_result(system='Linux', node='yichi-thinkpad', release='4.4.0-26100-Microsoft', version='#8115-Microsoft Fri Jan 01 08:00:00 PST 2016', machine='x86_64')  Threads 12
Python 3.10.16 | packaged by conda-forge | (main, Dec  5 2024, 14:16:10) [GCC 13.3.0]
numpy 1.24.3  scipy 1.14.1  h5py 3.12.1
Date: Thu May 14 11:19:26 2026
PySCF version 2.12.1
PySCF path  /home/yichi/research/software/pyscf
GIT ORIG_HEAD a0665c4a7bf54e33f01295b3eea390be7a17d76d
GIT HEAD (branch master) f97393b29b0a541c155a68d55ee5b652ae7131d2

[ENV] OLD_PYSCF_EXT_PATH 
[ENV] PYSCF_EXT_PATH /home/yichi/research/software/pyscf-forge:
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 3
[INPUT] num. electrons = 10
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry 0 subgroup None
[INPUT] Mole.unit = angstrom
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[INPUT]  1 O      0.000000000

In [33]:
from pyscf import gto, scf

# Read your xyz exactly as before
xyzfile = "/home/yichi/research/w4_17/w4-17_all/w4_17_xyz/closed_shell/h2o.xyz"
with open(xyzfile, "r") as f:
    lines = f.readlines()
    second_line = lines[1]
    charge = int(second_line.split("charge=")[1].split()[0])
    mult   = int(second_line.split("mult=")[1].split()[0])
    spin   = mult - 1
    atoms  = "".join(lines[2:])

h_basis = gto.basis.load('ccpvdz', 'H')
h_basis_no_p = [b for b in h_basis if b[0] != 1]

mol = gto.M(atom = atoms,
            basis = {'H': h_basis_no_p,
                     'default': 'ccpvdz'},   # custom per-element basis
            verbose=4,
            unit='angstrom',
            symmetry=0,
            charge=charge,
            spin=spin,
            max_memory=40000)

mf = scf.RHF(mol)
mf.max_cycle = 100
mf.kernel()

stable = False
while not stable:
    mo_i, _, stable, _ = mf.stability(return_status=True)
    dm = mf.make_rdm1(mo_i, mf.mo_occ)
    mf.kernel(dm0=dm)

mycc = cc.CCSD(mf)
mycc.set_frozen()
mycc.kernel()
et = mycc.ccsd_t()

print(f"E SCF-HF = {mf.e_tot}")
print(f"E CCSD   = {mycc.e_tot}")
print(f"E CCSD(T) = {mycc.e_tot+et}")

System: uname_result(system='Linux', node='yichi-thinkpad', release='4.4.0-26100-Microsoft', version='#8115-Microsoft Fri Jan 01 08:00:00 PST 2016', machine='x86_64')  Threads 12
Python 3.10.16 | packaged by conda-forge | (main, Dec  5 2024, 14:16:10) [GCC 13.3.0]
numpy 1.24.3  scipy 1.14.1  h5py 3.12.1
Date: Wed May 13 21:35:38 2026
PySCF version 2.12.1
PySCF path  /home/yichi/research/software/pyscf
GIT ORIG_HEAD a0665c4a7bf54e33f01295b3eea390be7a17d76d
GIT HEAD (branch master) f97393b29b0a541c155a68d55ee5b652ae7131d2

[ENV] OLD_PYSCF_EXT_PATH 
[ENV] PYSCF_EXT_PATH /home/yichi/research/software/pyscf-forge:
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 3
[INPUT] num. electrons = 10
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry 0 subgroup None
[INPUT] Mole.unit = angstrom
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[INPUT]  1 O      0.000000000

In [7]:
def vdzsd(elem):
    if elem in ('H', 'He'):
        h_basis = gto.basis.load('ccpvdz', elem)
        return [b for b in h_basis if b[0] != 1]
    else: 
        return gto.basis.load('ccpvdz', elem)

def get_vdzsd_basis(atoms):
    elems = {line.split()[0] for line in atoms.strip().splitlines() if line.strip()}
    basis_dict = {el: vdzsd(el) for el in elems}
    return basis_dict

In [11]:
basis = get_vdzsd_basis(atoms)
basis["O"]

[[0,
  [11720.0, 0.00071, -0.00016],
  [1759.0, 0.00547, -0.001263],
  [400.8, 0.027837, -0.006267],
  [113.7, 0.1048, -0.025716],
  [37.03, 0.283062, -0.070924],
  [13.27, 0.448719, -0.165411],
  [5.025, 0.270952, -0.116955],
  [1.013, 0.015458, 0.557368]],
 [0, [0.3023, 1.0]],
 [1, [17.7, 0.043018], [3.854, 0.228913], [1.046, 0.508728]],
 [1, [0.2753, 1.0]],
 [2, [1.185, 1.0]]]

In [10]:
gto.basis.load('ccpvdz', "O")

[[0,
  [11720.0, 0.00071, -0.00016],
  [1759.0, 0.00547, -0.001263],
  [400.8, 0.027837, -0.006267],
  [113.7, 0.1048, -0.025716],
  [37.03, 0.283062, -0.070924],
  [13.27, 0.448719, -0.165411],
  [5.025, 0.270952, -0.116955],
  [1.013, 0.015458, 0.557368]],
 [0, [0.3023, 1.0]],
 [1, [17.7, 0.043018], [3.854, 0.228913], [1.046, 0.508728]],
 [1, [0.2753, 1.0]],
 [2, [1.185, 1.0]]]

In [20]:
from pyscf import gto, scf

# Read your xyz exactly as before
xyzfile = "/home/yichi/research/w4_17/w4-17_all/w4_17_xyz/closed_shell/h2o.xyz"
with open(xyzfile, "r") as f:
    lines = f.readlines()
    second_line = lines[1]
    charge = int(second_line.split("charge=")[1].split()[0])
    mult   = int(second_line.split("mult=")[1].split()[0])
    spin   = mult - 1
    atoms  = "".join(lines[2:])

basis_dict = get_vdzsd_basis(atoms)

mol = gto.M(atom = atoms,
            basis = basis_dict,
            verbose=4,
            unit='angstrom',
            symmetry=True,
            charge=charge,
            spin=spin,
            max_memory=40000)

mf = scf.RHF(mol)
mf.max_cycle = 100
mf.kernel()

stable = False
while not stable:
    mo_i, _, stable, _ = mf.stability(return_status=True)
    dm = mf.make_rdm1(mo_i, mf.mo_occ)
    mf.kernel(dm0=dm)

mycc = cc.CCSD(mf)
mycc.set_frozen()
mycc.kernel()
et = mycc.ccsd_t()

print(f"E SCF-HF = {mf.e_tot}")
print(f"E CCSD   = {mycc.e_tot}")
print(f"E CCSD(T) = {mycc.e_tot+et}")

System: uname_result(system='Linux', node='yichi-thinkpad', release='4.4.0-26100-Microsoft', version='#8115-Microsoft Fri Jan 01 08:00:00 PST 2016', machine='x86_64')  Threads 12
Python 3.10.16 | packaged by conda-forge | (main, Dec  5 2024, 14:16:10) [GCC 13.3.0]
numpy 1.24.3  scipy 1.14.1  h5py 3.12.1
Date: Thu May 14 11:41:32 2026
PySCF version 2.12.1
PySCF path  /home/yichi/research/software/pyscf
GIT ORIG_HEAD a0665c4a7bf54e33f01295b3eea390be7a17d76d
GIT HEAD (branch master) f97393b29b0a541c155a68d55ee5b652ae7131d2

[ENV] OLD_PYSCF_EXT_PATH 
[ENV] PYSCF_EXT_PATH /home/yichi/research/software/pyscf-forge:
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 3
[INPUT] num. electrons = 10
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry True subgroup None
[INPUT] Mole.unit = angstrom
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[INPUT]  1 O      0.000000

In [14]:
from pyscf import gto, scf

# Read your xyz exactly as before
xyzfile = "/home/yichi/research/w4_17/w4-17_all/w4_17_xyz/closed_shell/bn.xyz"
with open(xyzfile, "r") as f:
    lines = f.readlines()
    second_line = lines[1]
    charge = int(second_line.split("charge=")[1].split()[0])
    mult   = int(second_line.split("mult=")[1].split()[0])
    spin   = mult - 1
    atoms  = "".join(lines[2:])

# Figure out which elements are actually in this molecule
# elements = {line.split()[0] for line in atoms.strip().splitlines() if line.strip()}
# basis_dict = {el: strip_polarization(el) for el in elements}

# basis_dict = get_vdzsd_basis(atoms)

mol = gto.M(atom = atoms,
            basis = "ccpvdz",
            verbose=4,
            unit='angstrom',
            symmetry=0,
            charge=charge,
            spin=spin,
            max_memory=40000)

mf = scf.RHF(mol)
mf.max_cycle = 100
mf.kernel()

stable = False
while not stable:
    mo_i, _, stable, _ = mf.stability(return_status=True)
    dm = mf.make_rdm1(mo_i, mf.mo_occ)
    mf.kernel(dm0=dm)

mycc = cc.CCSD(mf)
mycc.set_frozen()
mycc.kernel()
et = mycc.ccsd_t()

print(f"E SCF-HF = {mf.e_tot}")
print(f"E CCSD   = {mycc.e_tot}")
print(f"E CCSD(T) = {mycc.e_tot+et}")

System: uname_result(system='Linux', node='yichi-thinkpad', release='4.4.0-26100-Microsoft', version='#8115-Microsoft Fri Jan 01 08:00:00 PST 2016', machine='x86_64')  Threads 12
Python 3.10.16 | packaged by conda-forge | (main, Dec  5 2024, 14:16:10) [GCC 13.3.0]
numpy 1.24.3  scipy 1.14.1  h5py 3.12.1
Date: Thu May 14 11:30:48 2026
PySCF version 2.12.1
PySCF path  /home/yichi/research/software/pyscf
GIT ORIG_HEAD a0665c4a7bf54e33f01295b3eea390be7a17d76d
GIT HEAD (branch master) f97393b29b0a541c155a68d55ee5b652ae7131d2

[ENV] OLD_PYSCF_EXT_PATH 
[ENV] PYSCF_EXT_PATH /home/yichi/research/software/pyscf-forge:
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 2
[INPUT] num. electrons = 12
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry 0 subgroup None
[INPUT] Mole.unit = angstrom
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[INPUT]  1 B      0.000000000

In [19]:
from pyscf import gto, scf

# Read your xyz exactly as before
xyzfile = "/home/yichi/research/w4_17/w4-17_all/w4_17_xyz/closed_shell/bn.xyz"
with open(xyzfile, "r") as f:
    lines = f.readlines()
    second_line = lines[1]
    charge = int(second_line.split("charge=")[1].split()[0])
    mult   = int(second_line.split("mult=")[1].split()[0])
    spin   = mult - 1
    atoms  = "".join(lines[2:])

# Figure out which elements are actually in this molecule
# elements = {line.split()[0] for line in atoms.strip().splitlines() if line.strip()}
# basis_dict = {el: strip_polarization(el) for el in elements}

basis_dict = get_vdzsd_basis(atoms)

mol = gto.M(atom = atoms,
            basis = basis_dict,
            verbose=4,
            unit='angstrom',
            symmetry=0,
            charge=charge,
            spin=spin,
            max_memory=40000)

mf = scf.RHF(mol)
mf.max_cycle = 100
mf.kernel()

stable = False
while not stable:
    mo_i, _, stable, _ = mf.stability(return_status=True)
    dm = mf.make_rdm1(mo_i, mf.mo_occ)
    mf.kernel(dm0=dm)

mycc = cc.CCSD(mf)
mycc.set_frozen()
mycc.kernel()
et = mycc.ccsd_t()

print(f"E SCF-HF = {mf.e_tot}")
print(f"E CCSD   = {mycc.e_tot}")
print(f"E CCSD(T) = {mycc.e_tot+et}")

System: uname_result(system='Linux', node='yichi-thinkpad', release='4.4.0-26100-Microsoft', version='#8115-Microsoft Fri Jan 01 08:00:00 PST 2016', machine='x86_64')  Threads 12
Python 3.10.16 | packaged by conda-forge | (main, Dec  5 2024, 14:16:10) [GCC 13.3.0]
numpy 1.24.3  scipy 1.14.1  h5py 3.12.1
Date: Thu May 14 11:37:23 2026
PySCF version 2.12.1
PySCF path  /home/yichi/research/software/pyscf
GIT ORIG_HEAD a0665c4a7bf54e33f01295b3eea390be7a17d76d
GIT HEAD (branch master) f97393b29b0a541c155a68d55ee5b652ae7131d2

[ENV] OLD_PYSCF_EXT_PATH 
[ENV] PYSCF_EXT_PATH /home/yichi/research/software/pyscf-forge:
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 2
[INPUT] num. electrons = 12
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry 0 subgroup None
[INPUT] Mole.unit = angstrom
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[INPUT]  1 B      0.000000000

number of shells = 10
number of NR pGTOs = 52
number of NR cGTOs = 28
basis = {'B': [[0, [4570.0, 0.000696, -0.000139], [685.9, 0.005353, -0.001097], [156.5, 0.027134, -0.005444], [44.47, 0.10138, -0.021916], [14.48, 0.272055, -0.059751], [5.131, 0.448403, -0.138732], [1.898, 0.290123, -0.131482], [0.3329, 0.014322, 0.539526]], [0, [0.1043, 1.0]], [1, [6.001, 0.035481], [1.241, 0.198072], [0.3364, 0.50523]], [1, [0.09538, 1.0]], [2, [0.343, 1.0]]], 'N': [[0, [9046.0, 0.0007, -0.000153], [1357.0, 0.005389, -0.001208], [309.3, 0.027406, -0.005992], [87.73, 0.103207, -0.024544], [28.56, 0.278723, -0.067459], [10.21, 0.44854, -0.158078], [3.838, 0.278238, -0.121831], [0.7466, 0.01544, 0.549003]], [0, [0.2248, 1.0]], [1, [13.55, 0.039919], [2.917, 0.217169], [0.7973, 0.510319]], [1, [0.2185, 1.0]], [2, [0.817, 1.0]]]}
ecp = {}
CPU time:        20.89


******** <class 'pyscf.scf.hf.RHF'> ********
method = RHF
initial guess = minao
damping factor = 0
level_shift factor = 0
DIIS = <class 'pysc